# CMS Data Preparation

In this notebook, we will prepare the data for the CMS by transforming it into the required format and structure.
## Table of Contents
### [Locations Table](#locations)
- **[Country Boundaries (OSM)](#country-boundaries)**
- **[HydroBASINS](#hydro)**

## Setup

### Library import


In [1]:
import os

import geopandas as gpd

### Variables

In [2]:
DATA_PATH = "../data/processed"

### Utils

In [3]:
def create_sql_insert_queries(df, table_name):
    """
    Create SQL INSERT queries from a DataFrame.
    """
    queries = []
    for _, row in df.iterrows():
        columns = ", ".join([f'"{col}"' for col in df.columns])
        values = ", ".join([f"'{str(value)}'" for value in row])
        query = f"INSERT INTO {table_name} ({columns}) VALUES ({values});"
        queries.append(query)
    return queries

<a id='locations'></a>
## Locations

<a id='country-boundaries'></a>
### Country Boundaries (OSM)

**Read data**

In [7]:
gdf_countries = gpd.read_file(os.path.join(DATA_PATH, "countries_sahel.geojson"))
gdf_countries.head()

,name,bbox,ISO3,geometry
0,Senegal,"{ ""bbox"": [ -17.749868599999999, 12.2402663999...",SEN,"POLYGON ((-17.74987 14.74329, -17.74984 14.741..."
1,Mauritania,"{ ""bbox"": [ -17.238095900000001, 14.7209909, -...",MRT,"POLYGON ((-17.2381 20.66622, -17.23676 20.6625..."
2,Mali,"{ ""bbox"": [ -12.2402835, 10.147811000000001, 4...",MLI,"POLYGON ((-12.24028 14.76472, -12.23971 14.762..."
3,Burkina Faso,"{ ""bbox"": [ -5.5132070000000004, 9.41047179999...",BFA,"POLYGON ((-5.51321 10.43079, -5.51319 10.43066..."
4,Niger,"{ ""bbox"": [ 0.16896530000000001, 11.6936070999...",NER,"POLYGON ((0.16897 14.52207, 0.17333 14.50556, ..."


**Transform data**

In [8]:
gdf_countries.rename(columns={"ISO3": "code"}, inplace=True)
gdf_countries["type"] = "ADMIN_REGION"
gdf_countries = gdf_countries[["name", "code", "type", "bbox", "geometry"]]
gdf_countries.head()

,name,code,type,bbox,geometry
0,Senegal,SEN,ADMIN_REGION,"{ ""bbox"": [ -17.749868599999999, 12.2402663999...","POLYGON ((-17.74987 14.74329, -17.74984 14.741..."
1,Mauritania,MRT,ADMIN_REGION,"{ ""bbox"": [ -17.238095900000001, 14.7209909, -...","POLYGON ((-17.2381 20.66622, -17.23676 20.6625..."
2,Mali,MLI,ADMIN_REGION,"{ ""bbox"": [ -12.2402835, 10.147811000000001, 4...","POLYGON ((-12.24028 14.76472, -12.23971 14.762..."
3,Burkina Faso,BFA,ADMIN_REGION,"{ ""bbox"": [ -5.5132070000000004, 9.41047179999...","POLYGON ((-5.51321 10.43079, -5.51319 10.43066..."
4,Niger,NER,ADMIN_REGION,"{ ""bbox"": [ 0.16896530000000001, 11.6936070999...","POLYGON ((0.16897 14.52207, 0.17333 14.50556, ..."


**Save data**

CSV

In [13]:
gdf_countries.to_csv(os.path.join(DATA_PATH, "countries_sahel.csv"), index=False)

GeoJSON

In [9]:
gdf_countries.to_file(os.path.join(DATA_PATH, "countries_sahel_db.geojson"), driver="GeoJSON")

**Insert data to database**

In [15]:
# Example usage:
sql_queries = create_sql_insert_queries(gdf_countries, "admin_regions")
for q in sql_queries[:3]:
    print(q)

INSERT INTO admin_regions ("name", "code", "type", "bbox", "geometry") VALUES ('Senegal', 'SEN', 'ADMIN_REGION', '{ "bbox": [ -17.749868599999999, 12.240266399999999, -11.3459503, 16.691971200000001 ] }', 'POLYGON ((-17.7498686 14.7432872, -17.7498419 14.7410488, -17.7497395 14.7366653, -17.7495388 14.732285, -17.74924 14.72791, -17.7488433 14.7235424, -17.7483488 14.7191842, -17.7477567 14.7148376, -17.7470673 14.7105046, -17.7467065 14.7083864, -17.7462811 14.7061872, -17.7453983 14.7018875, -17.7444194 14.6976076, -17.7433448 14.6933496, -17.740968 14.68537, -17.7408214 14.6849433, -17.7401624 14.6828349, -17.7394174 14.6805421, -17.7387155 14.6784609, -17.738061 14.6765893, -17.7373197 14.6745422, -17.7364645 14.672261, -17.7356693 14.6702094, -17.7349314 14.668367, -17.7340915 14.6663353, -17.7331447 14.6641152, -17.7322576 14.6620974, -17.7314356 14.660283, -17.7305066 14.658291, -17.7294647 14.6561203, -17.7284335 14.6540345, -17.7245146 14.6462676, -17.7223453 14.6424251, -17.7